# Fase 3 · M02: Agregación por Expediente

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 3 — Feature Engineering |
| **Módulo** | M02 — Agregación |

---

## 🎯 Qué hace

Agrega el dataset a nivel de expediente académico, calculando variables de trayectoria (créditos, notas, años) por alumno.

## 📋 Requisitos

- `data/03_features/df_alumno_limpio.parquet`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/03_features/df_expediente_base.parquet` | Dataset agregado por expediente (42 cols) |

## 📋 Campos generados

| Grupo | Campos | Método |
|---|---|---|
| Identificadores | `per_id_ficticio`, `exp_tit_id` | primer registro |
| Temporales | `curso_inicio`, `curso_ultimo`, `n_cursos`, `anios_gap` | min/max/count/primer |
| Créditos | `cred_matriculados_total`, `cred_superados_total`, `cred_titulacion`, `cred_superados_anio_medio`, `cred_superados_anio_1er`, `tasa_rendimiento`, `cred_repetidos`, `tasa_repeticion` | sum/max/mean/calc |
| Notas | `media_global`, `nota_1er_anio`, `nota_ultimo_anio`, `nota_acceso`, `nota_selectividad` | mean/primer |
| Titulación | `titulacion`, `rama` | primer |
| Demográfico | `sexo`, `fecha_nacimiento`, `edad_entrada`, `pais_nombre`, `provincia`, `poblacion` | primer |
| Acceso | `via_acceso`, `orden_preferencia`, `cupo`, `universidad_origen` | primer |
| Beca | `n_anios_beca` | sum |
| Laboral | `situacion_laboral`, `n_anios_trabajando` | mode/sum |
| Económico | `max_pagos` | max |
| Estado ⚠️leakage | `egresado`, `egresado_de_hecho` | último/calc — M05 los elimina |
| Indicadores | `indicador_edad_inusual`, `indicador_interrupcion`, `indicador_sin_notas`, `n_anios_sin_notas` | any/all/sum |

## ⚠️ Campos eliminados respecto a versión anterior
| Campo eliminado | Motivo |
|---|---|
| `tuvo_beca` | Redundante con `n_anios_beca` |
| `pago_fraccionado` | Redundante con `max_pagos` |
| `indicador_casi_termino` | Todos False (campo muerto) + leakage |
| `mejora_notas` | Feature derivada — la calcula M03, no M02 |
| `docs/html/fase3/m02_agregacion.html` | Informe HTML |

## 🔄 Flujo

```
df_alumno_limpio.parquet
    ↓ Agrupación por per_id_ficticio
    ↓ Cálculo de variables de trayectoria
    → data/03_features/df_expediente_base.parquet + HTML
```

## ➡️ Siguiente

`f3_m03_features.ipynb` — generación de features temporales y derivadas


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')

# Detectar entorno
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import RUTA_FEATURES, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es, formato_porcentaje_es
from src.utils.graficos import histograma_con_kde, figura_a_base64, COLORES
from src.html import (
    generar_kpis_html,
    generar_seccion_html,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_pagina_desde_fichero

# Rutas
RUTA_FASE3_HTML = RUTA_HTML / 'fase3'
crear_directorios([RUTA_FEATURES, RUTA_FASE3_HTML])

info_entorno()

✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\FF\AU_UJI_v2
✓ 📁 RAW:           C:\FF\AU_UJI_v2\data\00_raw
✓ 📁 INTERIM:       C:\FF\AU_UJI_v2\data\01_interim
✓ 📁 PROCESSED:     C:\FF\AU_UJI_v2\data\02_processed
✓ 📁 FEATURES:      C:\FF\AU_UJI_v2\data\03_features
✓ 📁 AUTOML:        C:\FF\AU_UJI_v2\data\automl
✓ 📁 NOTEBOOKS:     C:\FF\AU_UJI_v2\notebooks
✓ 📄 Excel principal: C:\FF\AU_UJI_v2\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓ ===========================================================================


In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS
# ============================================================================

print('=' * 60)
print('F3-M02: AGREGACIÓN POR EXPEDIENTE')
print('=' * 60)

df = pd.read_parquet(RUTA_FEATURES / 'df_alumno_limpio.parquet')
fmt = formato_numero_es

n_registros = len(df)
n_expedientes = df.groupby(['per_id_ficticio', 'exp_tit_id']).ngroups

print(f'📥 Cargado: {fmt(n_registros)} registros (alumno×curso)')
print(f'📊 Expedientes únicos: {fmt(n_expedientes)}')
print(f'📈 Media registros/expediente: {n_registros/n_expedientes:.1f}')

F3-M02: AGREGACIÓN POR EXPEDIENTE
📥 Cargado: 109.568 registros (alumno×curso)
📊 Expedientes únicos: 33.621
📈 Media registros/expediente: 3.3


In [3]:
# ============================================================================
# CELDA 3: DEFINIR FUNCIÓN DE AGREGACIÓN
# ============================================================================

print('\n' + '=' * 60)
print('DEFINIENDO AGREGACIÓN')
print('=' * 60)

def agregar_expediente(g):
    """
    Agrega un grupo (expediente) a una sola fila.
    g: DataFrame con todos los registros de un expediente (per_id_ficticio + exp_tit_id)
    """
    # Ordenar por curso
    g = g.sort_values('curso_aca')
    
    # Cursos
    curso_inicio = g['curso_aca'].min()
    curso_ultimo = g['curso_aca'].max()
    n_cursos = g['curso_aca'].nunique()
    
    # Créditos
    cred_matriculados_total = g['cred_matriculados'].sum()  # por curso, se suma
    cred_superados_acum = g['cred_superados'].max()  # acumulativo, se toma max
    cred_superados_total = cred_superados_acum  # max porque es acumulativo
    
    # Notas
    notas_validas = g['media_curso'].dropna()
    media_global = notas_validas.mean() if len(notas_validas) > 0 else np.nan
    nota_1er_anio = g[g['curso_aca'] == curso_inicio]['media_curso'].mean()
    nota_ultimo_anio = g[g['curso_aca'] == curso_ultimo]['media_curso'].mean()
    
    # Primer registro (datos estáticos)
    primer = g.iloc[0]
    ultimo = g.iloc[-1]
    
    # --- Campos calculados ---
    cred_repetidos = max(0, cred_matriculados_total - primer['cred_titulacion'])
    tasa_repeticion = (cred_repetidos / primer['cred_titulacion'] * 100) if primer['cred_titulacion'] > 0 else 0
    n_anios_beca = (g['tiene_beca'] == True).sum() if 'tiene_beca' in g.columns else 0
    n_anios_trabajando = g['nombre_trabajo'].notna().sum() if 'nombre_trabajo' in g.columns else 0
    n_anios_sin_notas = (g['indicador_sin_notas'] == 1).sum() if 'indicador_sin_notas' in g.columns else 0

    return pd.Series({
        # Identificadores
        'per_id_ficticio': primer['per_id_ficticio'],
        'exp_tit_id': primer['exp_tit_id'],

        # Temporales
        'curso_inicio': curso_inicio,
        'curso_ultimo': curso_ultimo,
        'n_cursos': n_cursos,
        # anios_gap: años sin matricularse (0=trayectoria continua)
        # Calculado en M01 como (curso_ultimo - curso_inicio + 1) - n_cursos_reales
        'anios_gap': primer['anios_gap'] if 'anios_gap' in primer.index else 0,

        # Créditos
        'cred_matriculados_total': cred_matriculados_total,
        'cred_superados_total': cred_superados_total,
        'cred_titulacion': primer['cred_titulacion'],
        'cred_superados_anio_medio': g['cred_superados_anio'].mean() if 'cred_superados_anio' in g.columns else np.nan,
        'cred_superados_anio_1er': g[g['curso_aca'] == g['curso_aca'].min()]['cred_superados_anio'].iloc[0] if 'cred_superados_anio' in g.columns else np.nan,
        'tasa_rendimiento': (g['cred_superados_anio'].sum() / cred_matriculados_total * 100) if 'cred_superados_anio' in g.columns and cred_matriculados_total > 0 else np.nan,
        # cred_repetidos: créditos matriculados por encima de los necesarios (asignaturas repetidas)
        'cred_repetidos': cred_repetidos,
        # tasa_repeticion: % de créditos repetidos sobre el total de la carrera
        'tasa_repeticion': tasa_repeticion,

        # Notas
        'media_global': media_global,
        'nota_1er_anio': nota_1er_anio,
        'nota_ultimo_anio': nota_ultimo_anio,
        'nota_acceso': primer['nota_acceso'],
        'nota_selectividad': primer['nota_selectividad'] if 'nota_selectividad' in primer.index else np.nan,
        # mejora_notas: NO se calcula aquí.
        # Es una feature derivada (nota_ultimo - nota_1er) que calcula M03.
        # M02 solo agrega — M03 deriva features a partir del agregado.

        # Titulación
        'titulacion': primer['titulacion'],
        'rama': primer['rama'],

        # Demográfico
        'sexo': primer['sexo'],
        'fecha_nacimiento': primer['fecha_nacimiento'],
        'edad_entrada': primer['edad_entrada_calc'],
        'pais_nombre': primer['pais_nombre'],
        'provincia': primer['provincia'],
        'poblacion': primer['poblacion'],

        # Acceso (orden_preferencia: 0=sin preinscripción, 1-20=posición elegida)
        'via_acceso': primer['via_acceso'],
        'orden_preferencia': primer['orden_preferencia'] if 'orden_preferencia' in primer.index else 0,
        'cupo': primer['cupo'],
        'universidad_origen': primer['universidad_origen'],

        # Beca
        # tuvo_beca eliminado — redundante con n_anios_beca (si n_anios_beca > 0, tuvo beca)
        'n_anios_beca': n_anios_beca,

        # Laboral
        # situacion_laboral: valor más frecuente a lo largo del expediente
        'situacion_laboral': g['nombre_trabajo'].mode().iloc[0] if 'nombre_trabajo' in g.columns and g['nombre_trabajo'].notna().any() else np.nan,
        # n_anios_trabajando: años que compatibilizó estudios y trabajo
        'n_anios_trabajando': n_anios_trabajando,

        # Económico
        # pago_fraccionado eliminado — redundante con max_pagos (si max_pagos > 1, pagó fraccionado)
        'max_pagos': g['numero_pagos'].max() if 'numero_pagos' in g.columns and g['numero_pagos'].notna().any() else np.nan,

        # Estado final (leakage — M05 los elimina antes de exportar a D_strict)
        'egresado': ultimo['egresado'],
        'egresado_de_hecho': 1 if (cred_superados_total >= primer['cred_titulacion'] and str(ultimo['egresado']).upper() != 'S') else 0,

        # Indicadores
        'indicador_edad_inusual': g['indicador_edad_inusual'].any() if 'indicador_edad_inusual' in g.columns else False,
        'indicador_interrupcion': g['indicador_interrupcion'].any() if 'indicador_interrupcion' in g.columns else False,
        # indicador_casi_termino eliminado — todos False (campo muerto) + leakage
        # indicador_sin_notas: True solo si TODOS los años del alumno son sin nota
        'indicador_sin_notas': g['indicador_sin_notas'].all() if 'indicador_sin_notas' in g.columns else False,
        # n_anios_sin_notas: años matriculado sin nota (distinto de anios_gap que son años sin matricular)
        'n_anios_sin_notas': n_anios_sin_notas,
    })

print('✅ Función de agregación definida')


DEFINIENDO AGREGACIÓN
✅ Función de agregación definida


In [4]:
# ============================================================================
# CELDA 4: EJECUTAR AGREGACIÓN
# ============================================================================

print('\n' + '=' * 60)
print('EJECUTANDO AGREGACIÓN')
print('=' * 60)

from tqdm import tqdm
tqdm.pandas(desc='Agregando expedientes')

df_exp = df.groupby(['per_id_ficticio', 'exp_tit_id'], group_keys=False).progress_apply(agregar_expediente)
df_exp = df_exp.reset_index(drop=True)

n_exp_salida = len(df_exp)
n_cols_salida = len(df_exp.columns)

print(f'\n📤 Resultado: {fmt(n_exp_salida)} expedientes × {n_cols_salida} columnas')


EJECUTANDO AGREGACIÓN


Agregando expedientes:   0%|          | 0/33621 [00:00<?, ?it/s]

Agregando expedientes:   0%|          | 16/33621 [00:00<03:30, 159.64it/s]

Agregando expedientes:   0%|          | 36/33621 [00:00<03:03, 183.05it/s]

Agregando expedientes:   0%|          | 74/33621 [00:00<02:03, 271.90it/s]

...

Agregando expedientes: 100%|█████████▉| 33506/33621 [01:24<00:00, 536.93it/s]

Agregando expedientes: 100%|█████████▉| 33560/33621 [01:24<00:00, 532.87it/s]

Agregando expedientes: 100%|█████████▉| 33614/33621 [01:24<00:00, 534.24it/s]

Agregando expedientes: 100%|██████████| 33621/33621 [01:25<00:00, 392.28it/s]


📤 Resultado: 33.621 expedientes × 41 columnas


In [5]:
# ============================================================================
# CELDA 5: ESTADÍSTICAS DEL RESULTADO
# ============================================================================

print('\n' + '=' * 60)
print('ESTADÍSTICAS')
print('=' * 60)

# Cursos por expediente
stats_cursos = {
    'min': df_exp['n_cursos'].min(),
    'max': df_exp['n_cursos'].max(),
    'mean': df_exp['n_cursos'].mean(),
    'median': df_exp['n_cursos'].median()
}
print(f"📊 Cursos por expediente:")
print(f"   Rango: {stats_cursos['min']}-{stats_cursos['max']}")
print(f"   Media: {stats_cursos['mean']:.1f}")
print(f"   Mediana: {stats_cursos['median']:.0f}")

# Estado final del expediente
n_egresados = (df_exp['egresado'] == 'S').sum()
pct_egresados = n_egresados / n_exp_salida * 100
n_de_hecho = (df_exp['egresado_de_hecho'] == 1).sum()
pct_de_hecho = n_de_hecho / n_exp_salida * 100
n_total_terminaron = n_egresados + n_de_hecho
n_no_terminaron = n_exp_salida - n_total_terminaron

print(f"\n🎓 Estado final del expediente:")
print(f"   Egresados (título oficial): {fmt(n_egresados)} ({pct_egresados:.1f}%)")
print(f"   Completaron créditos sin título: {fmt(n_de_hecho)} ({pct_de_hecho:.1f}%)")
print(f"   Total terminaron: {fmt(n_total_terminaron)} ({n_total_terminaron/n_exp_salida*100:.1f}%)")
print(f"   No terminaron: {fmt(n_no_terminaron)} ({n_no_terminaron/n_exp_salida*100:.1f}%)")

print(f"\n📊 Rendimiento:")
if 'media_global' in df_exp.columns:
    print(f"   Nota media global: {df_exp['media_global'].mean():.2f}")
    print(f"   Nota media 1er año: {df_exp['nota_1er_anio'].mean():.2f}")
if 'cred_superados_total' in df_exp.columns:
    tasa_superacion = (df_exp['cred_superados_total'] / df_exp['cred_matriculados_total'].replace(0, np.nan)).mean() * 100
    print(f"   Tasa superación media: {tasa_superacion:.1f}%)")
    cred_medio = df_exp['cred_superados_total'].mean()
    print(f"   Créditos superados medio: {cred_medio:.0f}")

# Indicadores
print(f"\n📋 Indicadores:")
for ind in ['indicador_edad_inusual', 'indicador_interrupcion', 'indicador_casi_termino', 'indicador_sin_notas']:
    if ind in df_exp.columns:
        n = df_exp[ind].sum()
        pct = n / n_exp_salida * 100
        print(f"   {ind}: {fmt(n)} ({pct:.2f}%)")


ESTADÍSTICAS
📊 Cursos por expediente:
   Rango: 1-11
   Media: 3.3
   Mediana: 3

🎓 Estado final del expediente:
   Egresados (título oficial): 12.392 (36.9%)
   Completaron créditos sin título: 170 (0.5%)
   Total terminaron: 12.562 (37.4%)
   No terminaron: 21.059 (62.6%)

📊 Rendimiento:
   Nota media global: 7.00
   Nota media 1er año: 6.84
   Tasa superación media: 73.1%)
   Créditos superados medio: 147

📋 Indicadores:
   indicador_edad_inusual: 1 (0.00%)
   indicador_interrupcion: 1.021 (3.04%)
   indicador_sin_notas: 2.162 (6.43%)


In [6]:
# ============================================================================
# CELDA 6: GRÁFICOS
# ============================================================================

print('\n' + '=' * 60)
print('GENERANDO GRÁFICOS')
print('=' * 60)

# Gráfico 1: Distribución de cursos por expediente
fig_cursos = histograma_con_kde(
    df_exp['n_cursos'],
    titulo='Cursos matriculados por expediente',
    xlabel='Nº cursos',
    color=COLORES['primary'],
    bins=15
)
img_cursos = figura_a_base64(fig_cursos)
plt.close()

# Gráfico 2: Distribución de créditos superados
fig_creditos = histograma_con_kde(
    df_exp['cred_superados_total'],
    titulo='Créditos superados totales',
    xlabel='Créditos',
    color=COLORES['success'],
    bins=30
)
img_creditos = figura_a_base64(fig_creditos)
plt.close()

# Gráfico 3: Distribución de media global
fig_media = histograma_con_kde(
    df_exp['media_global'].dropna(),
    titulo='Media global por expediente',
    xlabel='Nota media',
    color=COLORES['warning'],
    bins=20
)
img_media = figura_a_base64(fig_media)
plt.close()

print('✅ Gráficos generados')


GENERANDO GRÁFICOS


✅ Gráficos generados


In [7]:
# ============================================================================
# CELDA 7: GUARDAR DATASET
# ============================================================================

print('\n' + '=' * 60)
print('GUARDANDO DATASET')
print('=' * 60)

ruta_salida = RUTA_FEATURES / 'df_expediente_base.parquet'
df_exp.to_parquet(ruta_salida, index=False)
tamanio_mb = ruta_salida.stat().st_size / 1024 / 1024
print(f'💾 Guardado: {ruta_salida.name} ({tamanio_mb:.1f} MB)')


GUARDANDO DATASET
💾 Guardado: df_expediente_base.parquet (1.2 MB)


In [8]:
# ============================================================================
# CELDA 8: GENERAR HTML
# ============================================================================

print('\n' + '=' * 60)
print('GENERANDO HTML')
print('=' * 60)

nav_fases_html, nav_modulos_html = generar_html_navegacion_completa(
    fase_activa='fase3',
    modulo_activo='m02'
)

# KPIs
KPIS = [
    {'valor': fmt(n_registros), 'titulo': 'Registros entrada'},
    {'valor': fmt(n_exp_salida), 'titulo': 'Expedientes'},
    {'valor': str(n_cols_salida), 'titulo': 'Columnas'},
    {'valor': f"{stats_cursos['mean']:.1f}", 'titulo': 'Media cursos'},
]
kpis_html = generar_kpis_html(KPIS)

# S1: Transformación
s1 = generar_seccion_html('Transformación', f'''
<div style="display:grid;grid-template-columns:1fr auto 1fr;gap:20px;align-items:center;text-align:center;">
    <div style="background:#ebf8ff;padding:20px;border-radius:10px;">
        <div style="font-size:28px;font-weight:bold;color:#3182ce;">{fmt(n_registros)}</div>
        <div style="color:#2c5282;">registros alumno×curso</div>
    </div>
    <div style="font-size:48px;color:#a0aec0;">→</div>
    <div style="background:#f0fff4;padding:20px;border-radius:10px;">
        <div style="font-size:28px;font-weight:bold;color:#38a169;">{fmt(n_exp_salida)}</div>
        <div style="color:#276749;">expedientes únicos</div>
    </div>
</div>
<p style="text-align:center;margin-top:15px;"><code>GROUP BY [per_id_ficticio, exp_tit_id]</code></p>
''', '🔄')

# S2: Variables agregadas
variables_agregadas = [
    # Temporales
    ('curso_inicio, curso_ultimo', 'min/max de curso_aca'),
    ('n_cursos', 'count distinct curso_aca'),
    ('anios_gap', 'primer registro — calculado en M01'),
    # Créditos
    ('cred_matriculados_total', 'sum(cred_matriculados)'),
    ('cred_superados_total', 'max(cred_superados) — acumulativo'),
    ('cred_superados_anio_medio', 'mean(cred_superados_anio)'),
    ('cred_superados_anio_1er', 'valor del primer año'),
    ('tasa_rendimiento', 'sum(cred_superados_anio) / cred_matriculados_total × 100'),
    ('cred_repetidos', 'max(0, cred_matriculados_total - cred_titulacion)'),
    ('tasa_repeticion', 'cred_repetidos / cred_titulacion × 100'),
    # Notas
    ('media_global', 'mean(media_curso) — ignorando NaN'),
    ('nota_1er_anio, nota_ultimo_anio', 'media del primer/último año'),
    # Beca y laboral
    ('n_anios_beca', 'sum(tiene_beca) — años con beca'),
    ('n_anios_trabajando', 'count(nombre_trabajo not null)'),
    ('situacion_laboral', 'mode(nombre_trabajo)'),
    # Económico
    ('max_pagos', 'max(numero_pagos)'),
    # Indicadores
    ('n_anios_sin_notas', 'sum(indicador_sin_notas)'),
    # Estado final — leakage, M05 los elimina
    ('egresado', 'último valor del expediente'),
    ('egresado_de_hecho', 'cred_superados >= cred_titulacion AND egresado != S'),
]

filas_vars = ''.join([f'<tr><td><code>{v}</code></td><td>{f}</td></tr>' for v, f in variables_agregadas])
s2 = generar_seccion_html('Variables Agregadas', f'''
<table style="width:100%;border-collapse:collapse;">
<tr style="background:#3182ce;color:white;"><th style="padding:10px;">Variable</th><th>Fórmula</th></tr>
{filas_vars}
</table>
''', '📊')

# S3: Estadísticas
s3 = generar_seccion_html('Estadísticas del Resultado', f'''
<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:15px;">
    <div style="background:#ebf8ff;padding:15px;border-radius:8px;text-align:center;">
        <div style="font-size:24px;font-weight:bold;color:#3182ce;">{stats_cursos["mean"]:.1f}</div>
        <div style="font-size:12px;color:#2c5282;">Cursos promedio</div>
    </div>
    <div style="background:#f0fff4;padding:15px;border-radius:8px;text-align:center;">
        <div style="font-size:24px;font-weight:bold;color:#38a169;">{pct_egresados:.1f}%</div>
        <div style="font-size:12px;color:#276749;">Egresados</div>
    </div>
    <div style="background:#fffaf0;padding:15px;border-radius:8px;text-align:center;">
        <div style="font-size:24px;font-weight:bold;color:#ed8936;">{(df_exp['egresado_de_hecho']==1).mean()*100:.1f}%</div>
        <div style="font-size:12px;color:#c05621;">Completaron sin título</div>
    </div>
    <div style="background:#fff5f5;padding:15px;border-radius:8px;text-align:center;">
        <div style="font-size:24px;font-weight:bold;color:#e53e3e;">{df_exp["media_global"].mean():.1f}</div>
        <div style="font-size:12px;color:#c53030;">Nota media</div>
    </div>
</div>
''', '📈')

# S4: Gráficos
s4 = generar_seccion_html('Distribuciones', f'''
<div style="display:grid;grid-template-columns:repeat(3,1fr);gap:20px;">
    <div style="text-align:center;"><img src="data:image/png;base64,{img_cursos}" style="max-width:100%;"/></div>
    <div style="text-align:center;"><img src="data:image/png;base64,{img_creditos}" style="max-width:100%;"/></div>
    <div style="text-align:center;"><img src="data:image/png;base64,{img_media}" style="max-width:100%;"/></div>
</div>
''', '📉')

# S5: Columnas del dataset por categorías
categorias_cols = {
    'Identificadores 🔑': ['per_id_ficticio', 'exp_tit_id'],
    'Temporal ⏱️': ['curso_inicio', 'curso_ultimo', 'n_cursos'],
    'Académico 🎓': ['cred_matriculados_total', 'cred_superados_total', 'cred_titulacion', 'media_global', 'nota_1er_anio', 'nota_ultimo_anio', 'nota_acceso', 'egresado'],
    'Titulación 📚': ['titulacion', 'rama'],
    'Demográfico 👤': ['sexo', 'fecha_nacimiento', 'edad_entrada', 'pais_nombre'],
    'Geográfico 🏠': ['provincia', 'poblacion'],
    'Acceso 📋': ['via_acceso', 'orden_preferencia', 'cupo', 'universidad_origen'],
    'Económico 💰': ['tuvo_beca', 'n_anios_beca'],
    'Indicadores 🏷️': ['indicador_edad_inusual', 'indicador_interrupcion', 'indicador_casi_termino', 'indicador_sin_notas'],
}

cats_html = ''
for cat, cols in categorias_cols.items():
    cols_existentes = [c for c in cols if c in df_exp.columns]
    if cols_existentes:
        cols_fmt = ', '.join([f'<code>{c}</code>' for c in cols_existentes])
        cats_html += f'''
        <div style="margin-bottom:15px;">
            <strong>{cat}</strong> ({len(cols_existentes)})
            <div style="margin-top:5px;color:#4a5568;line-height:1.8;">{cols_fmt}</div>
        </div>
        '''

s5 = generar_seccion_html('Columnas del Dataset', f'''
{cats_html}
<p style="margin-top:15px;padding:10px;background:#f7fafc;border-radius:5px;">
    <strong>Total:</strong> {n_cols_salida} columnas
</p>
''', '📋')

# HTML completo
contenido_html = kpis_html + s1 + s2 + s3 + s4 + s5

html_completo = render_pagina_desde_fichero(
    'f3_m02_agregacion.ipynb',
    contenido_html,
    carpeta_notebook='fase3_features'
)

ruta_html = RUTA_FASE3_HTML / 'm02_agregacion.html'
guardar_html(html_completo, ruta_html)
print(f'🌐 HTML: {ruta_html}')


GENERANDO HTML
✅ HTML guardado: C:\FF\AU_UJI_v2\docs\html\fase3\m02_agregacion.html
🌐 HTML: C:\FF\AU_UJI_v2\docs\html\fase3\m02_agregacion.html


In [9]:
# ============================================================================
# CELDA 9: RESUMEN FINAL
# ============================================================================

print('\n' + '=' * 60)
print('✅ F3-M02 COMPLETADO')
print('=' * 60)
print(f'📥 Entrada: {fmt(n_registros)} registros')
print(f'📤 Salida: {fmt(n_exp_salida)} expedientes × {n_cols_salida} columnas')
print(f'💾 {ruta_salida}')
print(f'🌐 {ruta_html}')
print(f'\n📌 Siguiente: f3_m03_features.ipynb')


✅ F3-M02 COMPLETADO
📥 Entrada: 109.568 registros
📤 Salida: 33.621 expedientes × 41 columnas
💾 C:\FF\AU_UJI_v2\data\03_features\df_expediente_base.parquet
🌐 C:\FF\AU_UJI_v2\docs\html\fase3\m02_agregacion.html

📌 Siguiente: f3_m03_features.ipynb
